# DPWH Infrastructure Verification using Satellite Imagery
## Weakly Supervised Geospatial Consistency Model

# 0. Install + Imports

In [1]:
import os
import numpy as np
import pandas as pd

import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader

from sklearn.model_selection import train_test_split
from sklearn.metrics import roc_auc_score

import geopandas as gpd
from shapely.geometry import Point

import osmnx as ox
import ee
import geemap

from tqdm import tqdm

KeyboardInterrupt: 

# 1. Google Earth Engine Authorization

In [ ]:
# ee.Authenticate()
# ee.Initialize()

KeyboardInterrupt: Interrupted by user

# 2. Load DPWH Dataset

In [14]:
from datasets import load_dataset

ds = load_dataset("bettergovph/dpwh-transparency-data")
raw_df = ds["train"].to_pandas()

display(raw_df)

,contractId,description,category,componentCategories,status,budget,amountPaid,progress,location,contractor,...,programName,sourceOfFunds,isLive,livestreamUrl,livestreamVideoId,livestreamDetectedAt,latitude,longitude,reportCount,hasSatelliteImage
0,21NA0052,"CONCRETING OF BRGY. MARAIGING FMR, BRGY. MARAI...",Roads,Roads,Completed,11939997.93,0,100.0,"{'province': 'Agusan del Norte DEO', 'region':...",C'ZARLES CONSTRUCTION & SUPPLY (23426),...,Outside Infra,Outside Infra - GAA 2021 DA FMR,False,NaN,NaN,NaN,9.367491,125.574854,0,True
1,23LF0026,BAGUAN BR. (B00119MN) ALONG SURIGAO DEL SUR BD...,Bridges,Bridges,Completed,4891483.94,0,100.0,"{'province': 'Davao Oriental 1st DEO', 'region...",ELLE-BUILDERS AND CONSTRUCTION SUPPLY (42245),...,Regular Infra,Regular Infra - GAA 2023 OO-1,False,NaN,NaN,NaN,7.034717,126.411111,0,True
2,25DI0029,OFF-CARRIAGEWAY IMPROVEMENT - TERTIARY ROADS -...,Roads,Roads,On-Going,36087410.00,0,0.0,"{'province': 'Laguna 2nd DEO', 'region': 'Regi...",MAGNA RICH TRADING & CONSTRUCTION (29108),...,Regular Infra,Regular Infra - GAA 2025 OO-1,False,NaN,NaN,NaN,14.173394,121.222769,0,True
3,24JE0007,BASIC INFRASTRUCTURE PROGRAM CONSTRUCTION OF R...,Roads,Roads,Completed,9889747.70,0,100.0,"{'province': 'Zamboanga del Sur 1st DEO', 'reg...",ZMS CONST. & DEV. CORP. (FORMERLY:ZMS CONSTRUC...,...,Regular Infra,Regular Infra - GAA 2024 SSP,False,NaN,NaN,NaN,8.036563,123.538867,0,True
4,20DJ0036,OO1: ENSURE SAFE AND RELIABLE NATIONAL ROAD SY...,Roads,Roads,Completed,28719880.00,0,100.0,"{'province': 'Quezon 2nd DEO', 'region': 'Regi...",ROMHEAD S.J. BUILDERS AND DEVELOPMENT INCORPOR...,...,Regular Infra,Regular Infra - GAA 2020 OO-1,False,NaN,NaN,NaN,13.930895,121.450798,0,True
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
248215,17RG0036,CLUSTER 3: CONSTRUCTION OF RAINWATER COLLECTOR...,Water Provision and Storage,Water Provision and Storage,Completed,1473096.00,0,100.0,"{'province': 'Negros Oriental 2nd DEO', 'regio...",GBTU CONSTRUCTION AND SUPPLIES (FOR: K.T.U. CO...,...,Regular Infra,Regular Infra - GAA 2016 MFO-3,False,NaN,NaN,NaN,9.733970,122.861907,0,False
248216,24AF0213,REHABILITATION OF DAMAGED SHORE PROTECTION WOR...,Flood Control and Drainage,NaN,For Procurement,0.00,0,0.0,"{'province': 'La Union 2nd DEO', 'region': 'Re...",NaN,...,NaN,NaN,False,NaN,NaN,NaN,NaN,NaN,1,False
248217,17AI0247,CONSTN OF ONE STRY TWO CL SB W/ TOILET ATTACHE...,Buildings and Facilities,Buildings and Facilities,Completed,2874249.72,0,100.0,"{'province': 'Pangasinan 3rd DEO', 'region': '...",2-R CONSTRUCTION AND SUPPLY (15304),...,Outside Infra,Outside Infra - GAA 2017 DepEd BEFF,False,NaN,NaN,NaN,16.027421,120.534282,0,False
248218,25IC0038,"CONSTRUCTION OF ROAD, BRGY. MILAGROSA TO BRGY....",Roads,Roads,Not Yet Started,12244896.84,0,0.0,"{'province': 'Leyte 1st DEO', 'region': 'Regio...",ZTATIK BUILDER INC. (43165),...,Regular Infra,Regular Infra - GAA 2025 SSP,False,NaN,NaN,NaN,11.173733,124.943061,0,False


In [18]:
raw_df.category.value_counts()

category
Buildings and Facilities                      91345
Roads                                         87524
Flood Control and Drainage                    33866
Bridges                                       12558
Water Provision and Storage                    6745
                                              ...  
GAA 2016 DOH HFEP                                 1
GAA 2023 FAPs                                     1
Flood Control: Construction - Dikes/Levees        1
GAA 2016 Foreign Assisted Projects                1
Dam: Construction - Concrete Dams                 1
Name: count, Length: 180, dtype: int64

In [33]:
raw_df.componentCategories.value_counts()

componentCategories
Buildings and Facilities                                                94014
Roads                                                                   91631
Flood Control and Drainage                                              34858
Bridges                                                                 13146
Water Provision and Storage                                              6767
Bridges, Roads                                                            963
Flood Control and Drainage, Roads                                         139
Buildings and Facilities, Roads                                            24
Roads, Water Provision and Storage                                         19
Bridges, Flood Control and Drainage                                        15
Bridges, Flood Control and Drainage, Roads                                 13
Buildings and Facilities, Roads, Water Provision and Storage               10
Buildings and Facilities, Flood Control and 

In [ ]:
df = (raw_df
        .map(lambda x: x.lower() if isinstance(x, str) else x)
        .assign(startDate = lambda x: pd.to_datetime(x.startDate, errors="coerce"),
                completionDate = lambda x: pd.to_datetime(x.completionDate, errors="coerce"),
                province = lambda x: x.location.str.get('province'),
                region = lambda x: x.location.str.get('region'),
                contractorId = lambda x: x.contractor.str.extract(r'\(.*?(\d+).*?\)')
                )
        .query(
                "category == 'roads' and "
                "status == 'completed' and "
                "description.str.contains('construction', case=False, na=False) and "
                "not description.str.contains('reconstruction', case=False, na=False)",
                engine='python'
            )
        .dropna(subset=["latitude", "longitude", "startDate", "completionDate"])
        .filter(['contractId', 'description', 'category', # project description
                'status', 'progress',  # project progess
                'region', 'province', 'latitude', 'longitude', # project location
                'startDate', 'completionDate', 'infraYear',  # project timeline
                'programName','sourceOfFunds', 'budget', 'amountPaid', 'contractorId' # project financing
                ])
        .copy()
    )


# df["label"] = (df["status"] == "completed").astype(int)

display(df, df.info(), df.infraYear.value_counts())

<class 'pandas.DataFrame'>
Index: 24236 entries, 3 to 248209
Data columns (total 17 columns):
 #   Column          Non-Null Count  Dtype        
---  ------          --------------  -----        
 0   contractId      24236 non-null  str          
 1   description     24236 non-null  str          
 2   category        24236 non-null  str          
 3   status          24236 non-null  str          
 4   progress        24236 non-null  float64      
 5   region          24236 non-null  object       
 6   province        24236 non-null  object       
 7   latitude        24236 non-null  float64      
 8   longitude       24236 non-null  float64      
 9   startDate       24236 non-null  datetime64[s]
 10  completionDate  24236 non-null  datetime64[s]
 11  infraYear       24236 non-null  str          
 12  programName     24236 non-null  str          
 13  sourceOfFunds   24236 non-null  str          
 14  budget          24236 non-null  float64      
 15  amountPaid      24236 non-null  in

,contractId,description,category,status,progress,region,province,latitude,longitude,startDate,completionDate,infraYear,programName,sourceOfFunds,budget,amountPaid,contractorId
3,24je0007,basic infrastructure program construction of r...,roads,completed,100.0,Region IX,Zamboanga del Sur 1st DEO,8.036563,123.538867,2024-02-19,2024-07-26,2024,regular infra,regular infra - gaa 2024 ssp,9889747.70,0,10471
7,21il0085,construction/ improvement of access roads lead...,roads,completed,100.0,Region VIII,Southern Leyte 1st DEO,10.330398,125.180188,2021-02-11,2021-10-08,2021,regular infra,regular infra - gaa 2021 cssp,44182320.00,0,5122
8,25mi0022,"construction of road, barangay capayangan, ban...",roads,completed,100.0,Region XII,Cotabato 3rd DEO,7.545831,124.745492,2025-05-23,2025-07-13,2025,regular infra,regular infra - gaa 2025 ssp,9893991.40,0,24433
11,24kl0087,"construction of road, sitio migbanday, baranga...",roads,completed,100.0,Region X,Misamis Oriental 2nd DEO,8.624808,124.926547,2024-03-07,2024-09-24,2024,regular infra,regular infra - gaa 2024 ssp,29700000.00,0,37020
16,23ka0099,construction/ improvement of access road leadi...,roads,completed,100.0,Region X,Bukidnon 1st DEO,8.181703,124.939863,2023-01-31,2023-08-23,2023,regular infra,regular infra - gaa 2023 cssp,19600000.00,0,6506
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
248088,15jb0132,"construction of manukan boulevard, manukan, za...",roads,completed,100.0,Region IX,Zamboanga del Norte 2nd DEO,8.522740,123.102946,2016-02-15,2017-01-15,2016,regular infra,regular infra - gaa 2016 locally funded projects,19173645.96,0,7024
248110,15dp0090,construction/upgrading/rehabilitation of drain...,roads,completed,100.0,Region IV-A,Laguna 3rd DEO,14.113261,121.416718,2016-01-20,2016-08-16,2016,regular infra,regular infra - gaa 2016 mfo-1,3193016.00,0,10722
248114,17il0051,off-carriageway improvement: shoulder paving/c...,roads,completed,100.0,Region VIII,Southern Leyte 1st DEO,10.126600,124.879530,2017-07-20,2017-12-16,2017,regular infra,regular infra - gaa 2017 lfp,4802191.27,0,40332
248197,17bb0013,mfo1-nat'l rd network services-network dev't-o...,roads,completed,100.0,Region II,Cagayan 1st DEO,17.932974,121.768427,2017-08-02,2019-04-01,2017,regular infra,regular infra - gaa 2017 mfo-1,46714718.41,0,11898


None

infraYear
2022    4555
2023    3768
2021    3190
2020    3048
2024    2961
2019    2674
2018    2493
2025     707
2017     458
2016     382
Name: count, dtype: int64